# LaMelo Brunson
This is my attempt at recreating FiveThirtyEight's discontinued "Carm-Elo," a backronym of basketball legend and best iso player of all time Camelo Anthony, that eventually was lengthened to "Career-Arc Regression Model Estimator with Local Optimization." This iteration will have the same goal of estimating player WAR for the next 5 years of their career.

This has two elements. The first of which is already made in a prior-made model that extrapolates Basketball Reference's Simple Projection System over 5 years.

The second of which generates similarity scores partitioned across age and uses the rest of the similar career to generate projections for the rest of their career.

The reason why I'm calling it "Lamelo Brunson" because Lamelo Ball and Jalen Brunson are the players who are making Carmelo Anthony irrelevant in history. Lamelo took his nickname "Melo" and Brunson has already become a better Knick than Melo, the Carmelo one, ever was. And Patrick Ewing, for that matter.

I'm going as far back as 1980, the first season of the NBA 3-pointer.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/nba-aba-baa-stats/Advanced.csv
/kaggle/input/nba-aba-baa-stats/Player Season Info.csv
/kaggle/input/nba-aba-baa-stats/Player Career Info.csv
/kaggle/input/nba-aba-baa-stats/Player Shooting.csv
/kaggle/input/nba-aba-baa-stats/Opponent Stats Per Game.csv
/kaggle/input/nba-aba-baa-stats/Player Totals.csv
/kaggle/input/nba-aba-baa-stats/Opponent Stats Per 100 Poss.csv
/kaggle/input/nba-aba-baa-stats/Per 100 Poss.csv
/kaggle/input/nba-aba-baa-stats/Team Summaries.csv
/kaggle/input/nba-aba-baa-stats/Player Per Game.csv
/kaggle/input/nba-aba-baa-stats/All-Star Selections.csv
/kaggle/input/nba-aba-baa-stats/End of Season Teams (Voting).csv
/kaggle/input/nba-aba-baa-stats/Player Play By Play.csv
/kaggle/input/nba-aba-baa-stats/End of Season Teams.csv
/kaggle/input/nba-aba-baa-stats/Team Stats Per 100 Poss.csv
/kaggle/input/nba-aba-baa-stats/Per 36 Minutes.csv
/kaggle/input/nba-aba-baa-stats/Opponent Totals.csv
/kaggle/input/nba-aba-baa-stats/Team Abbrev.csv
/kaggle/input/nba-aba-b

# Parameters
Shooting Statistical Categories:
- FGM
- FGA
- 3PM
- 3PA
- FTM
- FTA

Other Statistical Categories:
- REB
- AST
- STL
- BLK
- TO

Basic Fact Data:
- Age
- Height
- Weight
- Draft Position **(WHERE TO GET LEL)**

Consequential Additional Data: **(WHERE THE HECK DO I GET THIS)**
- Minutes per game
- Usage rate
- Box plus minus


The weighting will be 25% each category, and each sub-weight is, frankly, arbitrarily and intuitively decided. I'm not particularly sure what is going on, but we'll see what would happen.

An interesting point is that... I decided not to use positions! I think the position of a player, or rather the role, can probably be discerned by stats. Like I'd argue Michael Jordan is intuitively more like Kevin Durant than Tony Allen, even if Tony Allen is also regarded (haha WSB) as a 2 guard, and KD is more likely a 4. Alexa play "positions" by Ariana Grande.

# STEP 1: Aggregating Historical Data by Player and Age

In [2]:
player_per_game = pd.read_csv('/kaggle/input/nba-aba-baa-stats/Player Per Game.csv')
player_per_game = player_per_game[['season', 'player', 'age', 'mp_per_game', 'x2p_per_game', 'x3p_per_game', 'ft_per_game', 'trb_per_game', 'ast_per_game', 'stl_per_game', 'blk_per_game', 'tov_per_game']]
print(player_per_game.columns)

Index(['season', 'player', 'age', 'mp_per_game', 'x2p_per_game',
       'x3p_per_game', 'ft_per_game', 'trb_per_game', 'ast_per_game',
       'stl_per_game', 'blk_per_game', 'tov_per_game'],
      dtype='object')


In [3]:
historical_player_per_game = player_per_game[(player_per_game['season'] > 1979) & (player_per_game['season'] < 2024)]
historical_player_per_game = historical_player_per_game.drop_duplicates(subset=['player', 'age'], keep='first')

# Define the time range
past_years = 3
future_years = 4

# Function to transform the dataset
def transform_player_age_stats(df, past_years, future_years):
    # Initialize results
    rows = []

    # Iterate over each player-age pair in the original data
    for _, row in df.iterrows():
        player = row['player']
        age = int(row['age'])

        # Get the range of interest (past and future years relative to the current age)
        age_range = list(range(int(age) - past_years, int(age) + future_years + 1))
        
        # Filter data for this player and the age range
        player_data = df[(df['player'] == player) & (df['age'].isin(age_range))]

        # Fill missing years with zeros
        stats = player_data.set_index('age').reindex(age_range, fill_value=0)

        # Flatten the stats into a single row with appropriate column labels
        flat_row = {
            'player': player,
            'age': age,
            **{f'mp_per_game_year{a - age}': stats.at[a, 'mp_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'x2p_per_game_year{a - age}': stats.at[a, 'x2p_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'x3p_per_game_year{a - age}': stats.at[a, 'x3p_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'ft_per_game_year{a - age}': stats.at[a, 'ft_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'trb_per_game_year{a - age}': stats.at[a, 'trb_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'ast_per_game_year{a - age}': stats.at[a, 'ast_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'stl_per_game_year{a - age}': stats.at[a, 'stl_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'blk_per_game_year{a - age}': stats.at[a, 'blk_per_game'] if a in stats.index else 0 for a in age_range},
            **{f'tov_per_game_year{a - age}': stats.at[a, 'tov_per_game'] if a in stats.index else 0 for a in age_range},
        }
        rows.append(flat_row)

    return pd.DataFrame(rows)

# Transform the dataset
historical_players_vectors = transform_player_age_stats(historical_player_per_game, past_years, future_years)


print(historical_players_vectors[historical_players_vectors['player'] == 'Michael Jordan'])

               player  age  mp_per_game_year-3  mp_per_game_year-2  \
10054  Michael Jordan   39                 0.0                 0.0   
10490  Michael Jordan   38                 0.0                 0.0   
12261  Michael Jordan   34                39.3                37.7   
12705  Michael Jordan   33                 0.0                39.3   
13137  Michael Jordan   32                39.3                 0.0   
13541  Michael Jordan   31                38.8                39.3   
14329  Michael Jordan   29                39.0                37.0   
14711  Michael Jordan   28                40.2                39.0   
15092  Michael Jordan   27                40.4                40.2   
15464  Michael Jordan   26                40.0                40.4   
15825  Michael Jordan   25                25.1                40.0   
16169  Michael Jordan   24                38.3                25.1   
16508  Michael Jordan   23                 0.0                38.3   
16826  Michael Jorda

# STEP 2: Getting All Non-Rookie Current Players (counted as anyone who has played in the past 3 years)

In [4]:
# Filter and remove duplicates
curr_player_per_game = player_per_game[(player_per_game['season'] > 2021) & (player_per_game['season'] < 2025)]
curr_player_per_game = curr_player_per_game.drop_duplicates(subset=['player', 'age'], keep='first')

# Define the time range
past_years = 3
future_years = 4

# Function to transform the dataset
def transform_player_to_single_vector(df, past_years, future_years):
    players = df['player'].unique()
    rows = []

    for player in players:
        # Filter data for the current player
        player_data = df[df['player'] == player]

        # Map years relative to 2025
        age_range = list(range(-past_years, future_years + 1))
        season_mapping = {year: 2025 + year for year in age_range}

        # Create a flat row for the player
        flat_row = {'player': player}

        # Determine the player's age in 2025
        age_2025 = None
        for season in range(2022, 2025):
            season_data = player_data[player_data['season'] == season]
            if not season_data.empty:
                age_in_season = int(season_data['age'].values[0])
                age_2025 = age_in_season + (2025 - season)
                break  # Stop after finding the first valid age

        flat_row['age'] = age_2025 if age_2025 is not None else 0

        for rel_year, season in season_mapping.items():
            # Get stats for the season
            season_data = player_data[player_data['season'] == season]

            # Populate stats or fill with zeros for each stat
            flat_row[f'mp_per_game_year{rel_year}'] = (
                season_data['mp_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'x2p_per_game_year{rel_year}'] = (
                season_data['x2p_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'x3p_per_game_year{rel_year}'] = (
                season_data['x3p_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'ft_per_game_year{rel_year}'] = (
                season_data['ft_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'trb_per_game_year{rel_year}'] = (
                season_data['trb_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'ast_per_game_year{rel_year}'] = (
                season_data['ast_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'stl_per_game_year{rel_year}'] = (
                season_data['stl_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'blk_per_game_year{rel_year}'] = (
                season_data['blk_per_game'].values[0] if not season_data.empty else 0
            )
            flat_row[f'tov_per_game_year{rel_year}'] = (
                season_data['tov_per_game'].values[0] if not season_data.empty else 0
            )

        rows.append(flat_row)

    return pd.DataFrame(rows)

# Transform the dataset
current_players_vectors = transform_player_to_single_vector(curr_player_per_game, past_years, future_years)
print(current_players_vectors[current_players_vectors['player'] == 'Shai Gilgeous-Alexander'])
print(current_players_vectors[current_players_vectors['player'] == 'Victor Wembanyama'])

                      player  age  mp_per_game_year-3  x2p_per_game_year-3  \
501  Shai Gilgeous-Alexander   26                34.7                  6.9   

     x3p_per_game_year-3  ft_per_game_year-3  trb_per_game_year-3  \
501                  1.6                 5.9                  5.0   

     ast_per_game_year-3  stl_per_game_year-3  blk_per_game_year-3  ...  \
501                  5.9                  1.3                  0.8  ...   

     tov_per_game_year3  mp_per_game_year4  x2p_per_game_year4  \
501                   0                  0                   0   

     x3p_per_game_year4  ft_per_game_year4  trb_per_game_year4  \
501                   0                  0                   0   

     ast_per_game_year4  stl_per_game_year4  blk_per_game_year4  \
501                   0                   0                   0   

     tov_per_game_year4  
501                   0  

[1 rows x 74 columns]
                player  age  mp_per_game_year-3  x2p_per_game_year-3  \
555  

# STEP 3: Make Comparisons and Project
Not entirely sure which comparison system would work the best, but I'm going to do a few and expand on what I think

# Manhattan Distance

In [5]:
def find_top_10_similar(current_player_name, current_players_vectors, historical_players_vectors):
    # Filter for the current player's vector
    current_player = current_players_vectors[current_players_vectors['player'] == current_player_name]
    if current_player.empty:
        raise ValueError(f"Player {current_player_name} not found in current players dataset.")

    # Include all relevant parameters in the similarity calculation
    relevant_columns = [
        'mp_per_game_year-3', 'mp_per_game_year-2', 'mp_per_game_year-1',
        'x2p_per_game_year-3', 'x2p_per_game_year-2', 'x2p_per_game_year-1',
        'x3p_per_game_year-3', 'x3p_per_game_year-2', 'x3p_per_game_year-1',
        'ft_per_game_year-3', 'ft_per_game_year-2', 'ft_per_game_year-1',
        'trb_per_game_year-3', 'trb_per_game_year-2', 'trb_per_game_year-1',
        'ast_per_game_year-3', 'ast_per_game_year-2', 'ast_per_game_year-1',
        'stl_per_game_year-3', 'stl_per_game_year-2', 'stl_per_game_year-1',
        'blk_per_game_year-3', 'blk_per_game_year-2', 'blk_per_game_year-1',
        'tov_per_game_year-3', 'tov_per_game_year-2', 'tov_per_game_year-1'
    ]

    # Extract the current player's vector and age
    current_vector = current_player[relevant_columns].to_numpy()
    current_age = current_player['age'].values[0]

    # Exclude the current player and limit to players with similar ages
    historical_filtered = historical_players_vectors[
        (historical_players_vectors['player'] != current_player_name) &
        (abs(historical_players_vectors['age'] - current_age) <= 1)
    ]

    # Calculate similarity for all seasons of each historical player
    historical_filtered = historical_filtered.copy()
    historical_vectors = historical_filtered[relevant_columns].to_numpy()
    manhattan_distances = np.sum(np.abs(historical_vectors - current_vector), axis=1)
    historical_filtered['distance'] = manhattan_distances
    historical_filtered['similarity_score'] = 1 / (1 + manhattan_distances)

    # Keep only the most similar season for each player
    historical_filtered = historical_filtered.sort_values('distance').drop_duplicates(subset=['player'], keep='first')

    # Sort by distance and return the top 10 most similar players
    top_10_similar = historical_filtered.nsmallest(10, 'distance')

    return top_10_similar

# Example invocation
current_player_name = 'Shai Gilgeous-Alexander'
top_10_similar_players = find_top_10_similar(current_player_name, current_players_vectors, historical_players_vectors)

print(top_10_similar_players)

                  player  age  mp_per_game_year-3  mp_per_game_year-2  \
5137   Russell Westbrook   25                34.7                35.3   
14899       Chris Mullin   27                33.9                37.7   
8088         Dwyane Wade   25                34.9                38.6   
687         Devin Booker   25                35.0                35.9   
17499       Reggie Theus   26                34.4                34.6   
5748     Carmelo Anthony   27                34.5                38.2   
44        Brandon Ingram   25                33.9                34.3   
104         De'Aaron Fox   25                32.0                35.1   
1188        Bradley Beal   27                36.3                36.9   
3388       DeMar DeRozan   27                38.2                35.0   

       mp_per_game_year-1  mp_per_game_year0  mp_per_game_year1  \
5137                 34.9               30.7               34.4   
14899                36.3               40.4               41.

# Cosine Similarity

In [6]:
from sklearn.metrics.pairwise import cosine_similarity

def find_top_10_similar_cosine(current_player_name, current_players_vectors, historical_players_vectors):
    # Filter for the current player's vector
    current_player = current_players_vectors[current_players_vectors['player'] == current_player_name]
    if current_player.empty:
        raise ValueError(f"Player {current_player_name} not found in current players dataset.")

    # Include all relevant parameters in the similarity calculation
    relevant_columns = [
        'mp_per_game_year-3', 'mp_per_game_year-2', 'mp_per_game_year-1',
        'x2p_per_game_year-3', 'x2p_per_game_year-2', 'x2p_per_game_year-1',
        'x3p_per_game_year-3', 'x3p_per_game_year-2', 'x3p_per_game_year-1',
        'ft_per_game_year-3', 'ft_per_game_year-2', 'ft_per_game_year-1',
        'trb_per_game_year-3', 'trb_per_game_year-2', 'trb_per_game_year-1',
        'ast_per_game_year-3', 'ast_per_game_year-2', 'ast_per_game_year-1',
        'stl_per_game_year-3', 'stl_per_game_year-2', 'stl_per_game_year-1',
        'blk_per_game_year-3', 'blk_per_game_year-2', 'blk_per_game_year-1',
        'tov_per_game_year-3', 'tov_per_game_year-2', 'tov_per_game_year-1'
    ]

    # Extract the current player's vector and age
    current_vector = current_player[relevant_columns].to_numpy()
    current_age = current_player['age'].values[0]

    # Exclude the current player and limit to players with similar ages
    historical_filtered = historical_players_vectors[
        (historical_players_vectors['player'] != current_player_name) &
        (abs(historical_players_vectors['age'] - current_age) <= 1)
    ]

    # Calculate cosine similarity for all seasons of each historical player
    historical_filtered = historical_filtered.copy()
    historical_vectors = historical_filtered[relevant_columns].to_numpy()
    similarity_scores = cosine_similarity(current_vector, historical_vectors).flatten()
    historical_filtered['similarity_score'] = similarity_scores

    # Keep only the most similar season for each player
    historical_filtered = historical_filtered.sort_values('similarity_score', ascending=False).drop_duplicates(subset=['player'], keep='first')

    # Sort by similarity and return the top 10 most similar players
    top_10_similar = historical_filtered.nlargest(10, 'similarity_score')

    return top_10_similar

# Example invocation
current_player_name = 'Shai Gilgeous-Alexander'
top_10_similar_players = find_top_10_similar_cosine(current_player_name, current_players_vectors, historical_players_vectors)

print(top_10_similar_players)

               player  age  mp_per_game_year-3  mp_per_game_year-2  \
7642      Dwyane Wade   26                38.6                38.6   
6874     LeBron James   25                40.9                40.4   
9121      Kobe Bryant   26                38.3                41.5   
9298    Tracy McGrady   25                38.3                39.4   
9781    Allen Iverson   27                40.8                42.0   
687      Devin Booker   25                35.0                35.9   
4519     Kevin Durant   26                38.6                38.5   
14899    Chris Mullin   27                33.9                37.7   
15092  Michael Jordan   27                40.4                40.2   
15288   Clyde Drexler   27                38.0                37.8   

       mp_per_game_year-1  mp_per_game_year0  mp_per_game_year1  \
7642                 37.9               38.3               38.6   
6874                 37.7               39.0               38.8   
9121                 37.6   

# Euclidean Distance

In [7]:
import numpy as np

def find_top_10_similar_euclidean(current_player_name, current_players_vectors, historical_players_vectors):
    # Filter for the current player's vector
    current_player = current_players_vectors[current_players_vectors['player'] == current_player_name]
    if current_player.empty:
        raise ValueError(f"Player {current_player_name} not found in current players dataset.")

    # Include all relevant parameters in the similarity calculation
    relevant_columns = [
        'mp_per_game_year-3', 'mp_per_game_year-2', 'mp_per_game_year-1',
        'x2p_per_game_year-3', 'x2p_per_game_year-2', 'x2p_per_game_year-1',
        'x3p_per_game_year-3', 'x3p_per_game_year-2', 'x3p_per_game_year-1',
        'ft_per_game_year-3', 'ft_per_game_year-2', 'ft_per_game_year-1',
        'trb_per_game_year-3', 'trb_per_game_year-2', 'trb_per_game_year-1',
        'ast_per_game_year-3', 'ast_per_game_year-2', 'ast_per_game_year-1',
        'stl_per_game_year-3', 'stl_per_game_year-2', 'stl_per_game_year-1',
        'blk_per_game_year-3', 'blk_per_game_year-2', 'blk_per_game_year-1',
        'tov_per_game_year-3', 'tov_per_game_year-2', 'tov_per_game_year-1'
    ]

    # Extract the current player's vector and age
    current_vector = current_player[relevant_columns].to_numpy()
    current_age = current_player['age'].values[0]

    # Exclude the current player and limit to players with similar ages
    historical_filtered = historical_players_vectors[
        (historical_players_vectors['player'] != current_player_name) &
        (abs(historical_players_vectors['age'] - current_age) <= 1)
    ]

    # Calculate Euclidean distance for all seasons of each historical player
    historical_filtered = historical_filtered.copy()
    historical_vectors = historical_filtered[relevant_columns].to_numpy()
    euclidean_distances = np.sqrt(np.sum((historical_vectors - current_vector) ** 2, axis=1))
    historical_filtered['distance'] = euclidean_distances
    historical_filtered['similarity_score'] = 1 / (1 + euclidean_distances)

    # Keep only the most similar season for each player
    historical_filtered = historical_filtered.sort_values('distance').drop_duplicates(subset=['player'], keep='first')

    # Sort by distance and return the top 10 most similar players
    top_10_similar = historical_filtered.nsmallest(10, 'distance')

    return top_10_similar

# Example invocation
current_player_name = 'Shai Gilgeous-Alexander'
top_10_similar_players = find_top_10_similar_euclidean(current_player_name, current_players_vectors, historical_players_vectors)

print(top_10_similar_players)

                  player  age  mp_per_game_year-3  mp_per_game_year-2  \
687         Devin Booker   25                35.0                35.9   
14899       Chris Mullin   27                33.9                37.7   
5137   Russell Westbrook   25                34.7                35.3   
5748     Carmelo Anthony   27                34.5                38.2   
16814       Mark Aguirre   26                34.4                36.7   
8088         Dwyane Wade   25                34.9                38.6   
17499       Reggie Theus   26                34.4                34.6   
104         De'Aaron Fox   25                32.0                35.1   
3388       DeMar DeRozan   27                38.2                35.0   
17107    Kiki Vandeweghe   26                33.8                35.5   

       mp_per_game_year-1  mp_per_game_year0  mp_per_game_year1  \
687                  33.9               34.5               34.6   
14899                36.3               40.4               41.

# Similarity Score

The baseline has a weight of a similarity score 100. The rest of them will be added with a weight of their similarity score and then divided by the total weight of all players added in.

This machinery will only be applied on players with a positive similarity score and of the same age.